<a href="https://colab.research.google.com/github/Johnogunlola/MRes-AI/blob/MRes/UK_Flight_Delay_Extended_Analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.system('python uk_flight_delay_ml_pipeline.py')

512

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from datetime import datetime
from pathlib import Path
from collections import defaultdict


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.inspection import PartialDependenceDisplay


In [5]:
HAS_XGB, HAS_LGBM, HAS_CAT = False, False, False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    pass

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    pass

try:
    from catboost import CatBoostClassifier, Pool
    HAS_CAT = True
except Exception:
    pass

# Explainability
HAS_SHAP = False
try:
    import shap
    HAS_SHAP = True
except Exception:
    pass


In [6]:
N_JOBS = max(1, os.cpu_count() - 1)

DATA_PATH = "Flight Punctuality Statistics UK (2019-2024).csv"  # ensure file present
ARTIFACT_DIR = Path("./artifacts")
FIG_DIR = Path("./figs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [7]:
BINARY_THRESHOLD = 15.0  # mins
# Multi-class bands:
# Class 0 = (<=0), 1 = 1–30, 2 = 31–60, 3 = >60
MC_BINS = [-np.inf, 0, 30, 60, np.inf]
MC_LABELS = [0, 1, 2, 3]


In [8]:
TRAIN_END = pd.Timestamp("2022-12-01")
VAL_START, VAL_END = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-12-01")
TEST_START, TEST_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-01")

RANDOM_STATE = 42

In [9]:
def read_and_basic_clean(data_path: str) -> pd.DataFrame:
    """Load CSV, parse dates, clip/normalize percents, remove duplicates.

    Returns a DataFrame with:
      - _date (pd.Timestamp)
      - cleaned percentage columns
      - 'average_delay_mins' (float, may have NaNs)
    """
    df = pd.read_csv(data_path)
    # Strip columns
    df.columns = [c.strip() for c in df.columns]
    # Parse reporting period
    date_col = "reporting_period" if "reporting_period" in df.columns else None
    if date_col is None:
        # Fallback search
        cand = [c for c in df.columns if "date" in c.lower() or "period" in c.lower()]
        if cand:
            date_col = cand[0]
        else:
            raise ValueError("No date column found. Expect 'reporting_period' like 'YYYY/MM'.")

    def _parse(p):
        # Expect format like '2019/01' or '2019-01'
        try:
            return pd.to_datetime(p, format="%Y/%m")
        except Exception:
            return pd.to_datetime(p, errors="coerce")

    df["_date"] = df[date_col].map(_parse)
    df = df.loc[~df["_date"].isna()].copy()
    df.sort_values("_date", inplace=True)

    # Deduplicate rows (per airport-route-airline-direction-period)
    key_cols = [c for c in ["reporting_airport", "origin_destination", "airline_name",
                            "arrival_departure", "_date"] if c in df.columns]
    df.drop_duplicates(subset=key_cols, keep="first", inplace=True)

    # Identify percentage columns
    pct_cols = [c for c in df.columns if "percent" in c.lower()]
    # Clip percent values to [0, 100]
    for c in pct_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df[c] = df[c].clip(lower=0, upper=100)

    # Row-wise percent sanity: if sum is close to 100 (+/- 5), normalize to sum exactly 100
    if pct_cols:
        pct_sum = df[pct_cols].sum(axis=1)
        mask_norm = pct_sum.between(95, 105)
        df.loc[mask_norm, pct_cols] = df.loc[mask_norm, pct_cols].div(pct_sum[mask_norm], axis=0) * 100

    # Ensure average_delay_mins present
    avg_col = None
    for c in df.columns:
        if "average" in c.lower() and "delay" in c.lower():
            avg_col = c
            break
    if avg_col is None:
        raise ValueError("No 'average_delay_mins'-like column found. Please check your file.")

    df[avg_col] = pd.to_numeric(df[avg_col], errors="coerce")
    # Clip average delay outliers to [0, 360] minutes
    df[avg_col] = df[avg_col].clip(lower=0, upper=360)

    # Remove clearly non-operational rows: missing key identifiers or all percents=0 & avg_delay NaN
    essential = ["reporting_airport", "origin_destination", "airline_name", "arrival_departure"]
    for c in essential:
        if c in df.columns:
            df = df[df[c].notna()]
    if pct_cols:
        all_zero = (df[pct_cols].sum(axis=1) == 0)
        df = df[~(all_zero & df[avg_col].isna())]

    # Add convenience normalized aggregates
    df["pct_on_time"] = 0.0
    parts = [
        "flights_more_than_15_minutes_early_percent",
        "flights_15_minutes_early_to_1_minute_early_percent",
        "flights_0_to_15_minutes_late_percent"
    ]
    for p in parts:
        if p in df.columns:
            df["pct_on_time"] += df[p].fillna(0)

    late_bins = [
        "flights_between_16_and_30_minutes_late_percent",
        "flights_between_31_and_60_minutes_late_percent",
        "flights_between_61_and_120_minutes_late_percent",
        "flights_between_121_and_180_minutes_late_percent",
        "flights_between_181_and_360_minutes_late_percent",
        "flights_more_than_360_minutes_late_percent"
    ]
    df["pct_over_15"] = df[late_bins].fillna(0).sum(axis=1)

    # Traffic intensity proxy (if no flight counts exist): use pct_over_15 as a proxy magnitude
    # If a 'flights' count column existed we would compute actual intensity; here we construct a proxy.
    df["traffic_intensity_proxy"] = df["pct_over_15"]  # monotonic with late share

    return df, avg_col, pct_cols



In [10]:

def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add year, month, season, and time index (months since start)."""
    df["year"] = df["_date"].dt.year
    df["month"] = df["_date"].dt.month

    # Season: DJF=Winter, MAM=Spring, JJA=Summer, SON=Autumn
    def season(m):
        if m in (12, 1, 2):
            return "winter"
        if m in (3, 4, 5):
            return "spring"
        if m in (6, 7, 8):
            return "summer"
        return "autumn"
    df["season"] = df["month"].map(season)

    # Time index (months since min date)
    min_date = df["_date"].min()
    df["time_index"] = (df["_date"].dt.year - min_date.year) * 12 + (df["_date"].dt.month - min_date.month)
    return df


In [11]:
def build_group_key(df: pd.DataFrame) -> pd.Series:
    """Define the grouping key to compute lags/rolling stats in a leakage-safe way."""
    keys = []
    for c in ["reporting_airport", "origin_destination", "airline_name", "arrival_departure"]:
        if c in df.columns:
            keys.append(df[c].astype(str))
    if not keys:
        keys = [pd.Series(["ALL"] * len(df), index=df.index)]
    return pd.Series(pd.util.hash_pandas_object(pd.concat(keys, axis=1), index=False).astype(np.int64), index=df.index)


In [12]:
def add_lagged_and_rolling(df: pd.DataFrame, avg_col: str, pct_cols: list) -> pd.DataFrame:
    """Create lag and rolling features per airport-route-airline-direction group."""
    df = df.sort_values(["_date"]).copy()
    df["group_id"] = build_group_key(df)

    # Create lag features for average delay and some key percentages
    def _group_apply(group):
        group = group.sort_values("_date")
        # Lags
        group[f"{avg_col}_lag1"] = group[avg_col].shift(1)
        group[f"{avg_col}_lag3"] = group[avg_col].shift(3)

        # Rolling means (window includes only past values)
        group[f"{avg_col}_roll3"] = group[avg_col].shift(1).rolling(window=3, min_periods=1).mean()
        group[f"{avg_col}_roll6"] = group[avg_col].shift(1).rolling(window=6, min_periods=1).mean()

        # Late share aggregates
        if "pct_over_15" in group.columns:
            group["pct_over_15_lag1"] = group["pct_over_15"].shift(1)
            group["pct_over_15_roll3"] = group["pct_over_15"].shift(1).rolling(3, min_periods=1).mean()
            group["pct_over_15_roll6"] = group["pct_over_15"].shift(1).rolling(6, min_periods=1).mean()

        # Optional: rolling for 'pct_on_time'
        if "pct_on_time" in group.columns:
            group["pct_on_time_lag1"] = group["pct_on_time"].shift(1)
            group["pct_on_time_roll3"] = group["pct_on_time"].shift(1).rolling(3, min_periods=1).mean()

        return group

    df = df.groupby("group_id", group_keys=False).apply(_group_apply)

    # Final touch: these lag/rolling features will be NaN at the start of each series.
    # We'll impute later using training-set statistics only (handled by pipeline).
    return df


In [13]:
def create_targets(df: pd.DataFrame, avg_col: str):
    """Create Task 1 (binary) and Task 2 (multi-class) labels from average delay."""
    # Task 1 — Binary: 0=On-time <=15, 1=Delayed >15
    df["y_binary"] = np.where(df[avg_col] > BINARY_THRESHOLD, 1, 0)
    # Task 2 — Multi-class bands
    df["y_multi"] = pd.cut(df[avg_col], bins=MC_BINS, labels=MC_LABELS, right=True).astype(int)
    return df


In [14]:
def chronological_masks(df: pd.DataFrame):
    """Return boolean masks for train/val/test splits."""
    train_mask = (df["_date"] <= TRAIN_END)
    val_mask = (df["_date"] >= VAL_START) & (df["_date"] <= VAL_END)
    test_mask = (df["_date"] >= TEST_START) & (df["_date"] <= TEST_END)
    return train_mask, val_mask, test_mask


In [15]:
def rolling_time_series_folds(dates: pd.Series, n_splits: int = 6, gap: int = 0):
    """
    Expanding-window time-series CV generator.
    Splits on chronological order of `dates`.
    """
    # Sort indices by date
    order = np.argsort(dates.values)
    idx_sorted = dates.index.values[order]
    unique_months = np.unique(dates.dt.to_period("M").astype(str))
    # Split months into n_splits+1 segments, using earlier segments as train and next as val
    # We ensure progressive expansion.
    month_splits = np.array_split(np.arange(len(unique_months)), n_splits + 1)
    for i in range(n_splits):
        train_month_idx = np.concatenate(month_splits[: i + 1])
        val_month_idx = month_splits[i + 1]
        train_months = unique_months[train_month_idx]
        val_months = unique_months[val_month_idx]
        train_idx = dates.index[dates.dt.to_period("M").astype(str).isin(train_months)]
        val_idx = dates.index[dates.dt.to_period("M").astype(str).isin(val_months)]
        # Optional gap
        if gap > 0:
            # remove last `gap` months from train
            if len(train_months) > gap:
                gap_months = train_months[-gap:]
                train_idx = train_idx[~train_idx.isin(dates.index[dates.dt.to_period("M").astype(str).isin(gap_months)])]
        yield train_idx.values, val_idx.values


In [16]:
def metric_summary_binary(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) == 2 else np.nan,
    }


In [17]:
def metric_summary_multiclass(y_true, y_pred):
    # macro metrics
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


In [18]:
def ci_from_folds(values):
    """Mean ± 95% CI from fold scores."""
    arr = np.asarray(values, dtype=float)
    mean = np.nanmean(arr)
    std = np.nanstd(arr, ddof=1)
    n = np.sum(~np.isnan(arr))
    ci = 1.96 * std / np.sqrt(max(n, 1))
    return mean, (mean - ci, mean + ci)


In [19]:
def get_feature_groups(df: pd.DataFrame, avg_col: str, pct_cols: list):
    """Return lists of numeric and categorical predictors for the model."""
    # Base categorical
    cat_cols = [c for c in ["reporting_airport", "origin_destination_country",
                            "origin_destination", "airline_name", "arrival_departure",
                            "season"] if c in df.columns]

    # Numeric predictors (exclude target and date)
    base_exclude = set([avg_col, "y_binary", "y_multi", "_date", "group_id"])
    numeric_candidates = [
        c for c in df.columns
        if c not in base_exclude and
           c not in cat_cols and
           (np.issubdtype(df[c].dtype, np.number))
    ]
    # Ensure percentage, lag, rolling, proxy, time_index included if numeric
    num_cols = numeric_candidates

    return num_cols, cat_cols


In [20]:
def make_preprocessor(num_cols, cat_cols, scale_numeric=False):
    """ColumnTransformer with leakage-safe imputers; optional scaling."""
    num_imputer = SimpleImputer(strategy="median")
    if scale_numeric:
        num_pipe = Pipeline([("imputer", num_imputer), ("scaler", StandardScaler())])
    else:
        num_pipe = Pipeline([("imputer", num_imputer)])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols)
        ],
        remainder="drop",
        n_jobs=N_JOBS
    )
    return pre


In [21]:
def compute_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return dict(zip(classes, weights))

In [22]:
def fit_with_cv(model_name, estimator, X, y, dates, task="binary", param_grid=None):
    """
    Fit estimator with rolling time-series CV on training+validation (or only training),
    return fold metrics and the best estimator refit on full train (chronological).
    """
    cv_metrics = defaultdict(list)
    best_params = None
    best_score = -np.inf
    best_estimator = None

    # We use our rolling fold generator (expanding window) for robustness.
    n_splits = 6
    for fold, (tr_idx, va_idx) in enumerate(rolling_time_series_folds(dates, n_splits=n_splits, gap=0), 1):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va, y_va = X[va_idx], y[va_idx]

        # If param grid provided, do a small grid search only on this fold (to avoid leakage)
        est = estimator
        if param_grid:
            gs = GridSearchCV(est, param_grid=param_grid, cv=[(np.arange(len(X_tr)), np.arange(len(X_tr)))],
                              scoring="f1" if task=="binary" else "f1_macro",
                              n_jobs=N_JOBS, verbose=0)
            gs.fit(X_tr, y_tr)
            est = gs.best_estimator_
        else:
            est.fit(X_tr, y_tr)

        # Evaluate
        if task == "binary":
            if hasattr(est, "predict_proba"):
                y_proba = est.predict_proba(X_va)[:, 1]
            elif hasattr(est, "decision_function"):
                scores = est.decision_function(X_va)
                y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
            else:
                y_proba = np.zeros_like(y_va, dtype=float)
            y_pred = est.predict(X_va)
            fold_metrics = metric_summary_binary(y_va, y_pred, y_proba)
            score = fold_metrics["f1"]
        else:
            y_pred = est.predict(X_va)
            fold_metrics = metric_summary_multiclass(y_va, y_pred)
            score = fold_metrics["f1_macro"]

        # Store
        for k, v in fold_metrics.items():
            cv_metrics[k].append(v)

        # Track best model by fold score (simple heuristic)
        if score > best_score:
            best_score = score
            best_params = getattr(est, "get_params", lambda: {})()

    # Finally, refit the estimator on the full data chronologically
    # (Using the last fold's best params if available)
    final_est = estimator.set_params(**{k: v for k, v in best_params.items()
                                        if not isinstance(v, (np.ndarray, pd.DataFrame))}) if best_params else estimator
    final_est.fit(X, y)
    return dict(cv_metrics), final_est


In [23]:
def plot_cv_boxplots(cv_store, task_name):
    """Boxplots of CV scores across models."""
    df = []
    for model, metrics in cv_store.items():
        for k, vals in metrics.items():
            # we will plot only core metrics
            if k in ["f1", "roc_auc", "f1_macro", "accuracy"]:
                for v in vals:
                    df.append({"model": model, "metric": k, "value": v})
    if not df:
        return
    df = pd.DataFrame(df)
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x="metric", y="value", hue="model")
    plt.title(f"Rolling CV scores — {task_name}")
    plt.ylabel("Score")
    plt.xlabel("Metric")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"cv_boxplots_{task_name}.png", dpi=300)
    plt.close()


In [24]:
def plot_confusion_and_reports(y_true, y_pred, labels, title, prefix):
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize=None)
    cm_norm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")
    # Confusion matrix (counts)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title + " — Confusion Matrix (Counts)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{prefix}_cm_counts.png", dpi=300)
    plt.close()
    # Confusion matrix (normalized)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title + " — Confusion Matrix (Normalized)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{prefix}_cm_normalized.png", dpi=300)
    plt.close()
    # Classification report heatmap
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    rpt_df = pd.DataFrame(report).T
    plt.figure(figsize=(8, 6))
    sns.heatmap(rpt_df.iloc[:-1, :], annot=True, fmt=".2f", cmap="Purples")
    plt.title(title + " — Classification Report")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{prefix}_report_heatmap.png", dpi=300)
    plt.close()


In [25]:
def plot_roc_pr_curves(models, X_test, y_test, task_name, prefix):
    if task_name != "binary":
        return
    plt.figure(figsize=(7, 6))
    for name, est in models.items():
        if hasattr(est, "predict_proba"):
            y_proba = est.predict_proba(X_test)[:, 1]
        elif hasattr(est, "decision_function"):
            scores = est.decision_function(X_test)
            y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        else:
            continue
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC curves — Binary delay detection")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{prefix}_roc.png", dpi=300)
    plt.close()

    # PR curves
    plt.figure(figsize=(7, 6))
    for name, est in models.items():
        if hasattr(est, "predict_proba"):
            y_proba = est.predict_proba(X_test)[:, 1]
        elif hasattr(est, "decision_function"):
            scores = est.decision_function(X_test)
            y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        else:
            continue
        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        f1 = f1_score(y_test, (y_proba >= 0.5).astype(int))
        plt.plot(rec, prec, label=f"{name} (F1@0.5={f1:.3f})")
    plt.title("Precision–Recall curves — Binary delay detection")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{prefix}_pr.png", dpi=300)
    plt.close()


In [26]:
def plot_correlation_heatmap(df, num_cols, prefix):
    corr = df[num_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Feature correlation heatmap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{prefix}_corr_heatmap.png", dpi=300)
    plt.close()


In [27]:
def model_list_binary(class_weight_dict):
    models = {}
    # Logistic Regression (baseline, interpretable)
    models["LogReg"] = LogisticRegression(
        class_weight=class_weight_dict, max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE
    )
    # Random Forest
    models["RF"] = RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_split=2, min_samples_leaf=1,
        n_jobs=N_JOBS, class_weight=class_weight_dict, random_state=RANDOM_STATE
    )
    # XGBoost
    if HAS_XGB:
        models["XGB"] = XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=N_JOBS,
            eval_metric="logloss", tree_method="hist"
        )
    # LightGBM
    if HAS_LGBM:
        models["LGBM"] = LGBMClassifier(
            n_estimators=600, num_leaves=63, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=N_JOBS
        )
    # CatBoost
    if HAS_CAT:
        models["CAT"] = CatBoostClassifier(
            depth=6, learning_rate=0.05, iterations=600, l2_leaf_reg=3.0,
            random_state=RANDOM_STATE, loss_function="Logloss", verbose=False,
            class_weights=None  # catboost handles imbalance differently
        )
    # MLP
    models["MLP"] = MLPClassifier(
        hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
        alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=100,
        random_state=RANDOM_STATE
    )
    return models

In [28]:
def model_list_multiclass(class_weight_dict):
    models = {}
    models["LogReg"] = LogisticRegression(
        class_weight=class_weight_dict, max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE,
        multi_class="auto"
    )
    models["RF"] = RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_split=2, min_samples_leaf=1,
        n_jobs=N_JOBS, class_weight=class_weight_dict, random_state=RANDOM_STATE
    )
    if HAS_XGB:
        models["XGB"] = XGBClassifier(
            n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=N_JOBS,
            eval_metric="mlogloss", tree_method="hist"
        )
    if HAS_LGBM:
        models["LGBM"] = LGBMClassifier(
            n_estimators=700, num_leaves=63, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=N_JOBS
        )
    if HAS_CAT:
        models["CAT"] = CatBoostClassifier(
            depth=6, learning_rate=0.05, iterations=700, l2_leaf_reg=3.0,
            random_state=RANDOM_STATE, loss_function="MultiClass", verbose=False
        )
    models["MLP"] = MLPClassifier(
        hidden_layer_sizes=(256, 128), activation="relu", solver="adam",
        alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=120,
        random_state=RANDOM_STATE
    )
    return models

In [29]:
def stacked_ensemble(base_models, meta_logreg):
    estimators = [(name, mdl) for name, mdl in base_models.items()]
    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_logreg,
        stack_method="predict_proba",
        n_jobs=N_JOBS,
        passthrough=False
    )
    return stack


In [30]:
def get_fitted_preprocessor(pre, X_train):
    """Fit preprocessor only on training set to avoid leakage and return it."""
    pre.fit(X_train)
    return pre


In [31]:
def get_feature_names(preprocessor):
    """Retrieve feature names from ColumnTransformer+OneHotEncoder for explainability."""
    out = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "remainder" and trans == "drop":
            continue
        if hasattr(trans, "get_feature_names_out"):
            try:
                feats = trans.get_feature_names_out(cols)
            except Exception:
                feats = cols
        else:
            feats = cols
        out.extend(list(feats))
    return out


In [32]:

# 1) Load & clean
df, avg_col, pct_cols = read_and_basic_clean(DATA_PATH)
print(f"Loaded shape: {df.shape}")
print(f"Average delay column: {avg_col}")
print(f"Percentage columns (n={len(pct_cols)}): {pct_cols[:5]}{' ...' if len(pct_cols)>5 else ''}")
print("Date range:", df["_date"].min(), "→", df["_date"].max())


Loaded shape: (316956, 20)
Average delay column: average_delay_mins
Percentage columns (n=9): ['flights_more_than_15_minutes_early_percent', 'flights_15_minutes_early_to_1_minute_early_percent', 'flights_0_to_15_minutes_late_percent', 'flights_between_16_and_30_minutes_late_percent', 'flights_between_31_and_60_minutes_late_percent'] ...
Date range: 2019-01-01 00:00:00 → 2024-12-01 00:00:00


In [33]:

# 2) Remove rows with missing or invalid average delay (for label creation)
df = df[~df[avg_col].isna()].copy()
print("After removing missing target:", df.shape)


After removing missing target: (316956, 20)


In [34]:

# 3) Temporal features
df = add_temporal_features(df)
print("Added temporal features: ['year', 'month', 'season', 'time_index']")


Added temporal features: ['year', 'month', 'season', 'time_index']


In [35]:

# 4) Leakage-safe lags & rolling
df = add_lagged_and_rolling(df, avg_col, pct_cols)
print("Added lag/rolling features (e.g., lag1, lag3, roll3, roll6)")


Added lag/rolling features (e.g., lag1, lag3, roll3, roll6)


In [36]:

# 5) Targets
df = create_targets(df, avg_col)
print("Targets created: y_binary, y_multi (classes:", sorted(df["y_multi"].unique()), ")")


Targets created: y_binary, y_multi (classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)] )


In [37]:

# Keep rows where lag1 exists
lag1_col = f"{avg_col}_lag1"
if lag1_col in df.columns:
    before = df.shape[0]
    df = df[~df[lag1_col].isna()].copy()
    print(f"Filtered rows without {lag1_col}: {before} → {df.shape[0]}")
else:
    print("Warning: lag1 column not found; no filtering applied.")


Filtered rows without average_delay_mins_lag1: 316956 → 294044


In [38]:

# 6) Feature groups
num_cols, cat_cols = get_feature_groups(df, avg_col, pct_cols)
print("Numeric features (sample):", num_cols[:8], "...")
print("Categorical features:", cat_cols)


Numeric features (sample): ['flights_more_than_15_minutes_early_percent', 'flights_15_minutes_early_to_1_minute_early_percent', 'flights_0_to_15_minutes_late_percent', 'flights_between_16_and_30_minutes_late_percent', 'flights_between_31_and_60_minutes_late_percent', 'flights_between_61_and_120_minutes_late_percent', 'flights_between_121_and_180_minutes_late_percent', 'flights_between_181_and_360_minutes_late_percent'] ...
Categorical features: ['reporting_airport', 'origin_destination_country', 'origin_destination', 'airline_name', 'arrival_departure', 'season']


In [39]:
# --- EDA Visuals ---

# Class distributions (binary)
plt.figure(figsize=(6, 4))
sns.countplot(x="y_binary", data=df)
plt.title("Binary class distribution (delay > 15 mins)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_dist_binary.png", dpi=300)
plt.close()

# Class distributions (multi-class)
plt.figure(figsize=(6, 4))
sns.countplot(x="y_multi", data=df, order=sorted(df["y_multi"].unique()))
plt.title("Multi-class severity distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_dist_multiclass.png", dpi=300)
plt.close()

# Delay distribution over time
plt.figure(figsize=(10, 4))
df_month = df.groupby(df["_date"].dt.to_period("M"))[avg_col].mean().astype(float)
df_month.index = df_month.index.to_timestamp()
df_month.plot()
plt.title("Average delay (mins) over time")
plt.xlabel("Month")
plt.ylabel("Avg delay (mins)")
plt.tight_layout()
plt.savefig(FIG_DIR / "avg_delay_over_time.png", dpi=300)
plt.close()

# Airport-level delay trends (top 6 by volume)
if "reporting_airport" in df.columns:
    top_airports = df["reporting_airport"].value_counts().head(6).index.tolist()
    plt.figure(figsize=(10, 6))
    for ap in top_airports:
        s = (df[df["reporting_airport"] == ap]
             .groupby(df["_date"].dt.to_period("M"))[avg_col].mean().astype(float))
        s.index = s.index.to_timestamp()
        plt.plot(s.index, s.values, label=ap)
    plt.title("Airport-level average delay trends")
    plt.xlabel("Month")
    plt.ylabel("Avg delay (mins)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "airport_delay_trends.png", dpi=300)
    plt.close()

# Correlation heatmap for numeric features
plot_correlation_heatmap(df, num_cols=[c for c in num_cols if df[c].dtype != "O"], prefix="eda")

In [40]:
# 7) Chronological split masks
train_mask, val_mask, test_mask = chronological_masks(df)
print("Split sizes:",
      "Train:", train_mask.sum(),
      "Val:", val_mask.sum(),
      "Test:", test_mask.sum())

Split sizes: Train: 175663 Val: 55122 Test: 63259


In [41]:
# 8) Prepare X/y for both tasks and identical feature sets
features = num_cols + cat_cols
df_model = df[features + ["y_binary", "y_multi", "_date"]].copy()

# Combined train+val (chronologically sorted, 0-indexed)
df_train_val = df_model[train_mask | val_mask].sort_values("_date").reset_index(drop=True)

X_trva_df = df_train_val[features].copy()
y_trva_bin_full = df_train_val["y_binary"].values.astype(int)
y_trva_mc_full = df_train_val["y_multi"].values.astype(int)
dates_trva_full = df_train_val["_date"]  # for rolling folds (0-indexed)

# Separate test
X_test_df = df_model[test_mask][features].copy()
yte_bin = df_model[test_mask]["y_binary"].values.astype(int)
yte_mc = df_model[test_mask]["y_multi"].values.astype(int)

print("Train+Val:", X_trva_df.shape, "Test:", X_test_df.shape)

Train+Val: (230785, 30) Test: (63259, 30)


In [42]:
# Prepare X_train exclusively for fitting preprocessors
X_train_for_preprocessor_fit = df_model[train_mask][features].copy()

# 9) Preprocessors
pre_tree = make_preprocessor(
    num_cols=[c for c in num_cols if c in X_train_for_preprocessor_fit.columns],
    cat_cols=[c for c in cat_cols if c in X_train_for_preprocessor_fit.columns],
    scale_numeric=False
)
pre_scaled = make_preprocessor(
    num_cols=[c for c in num_cols if c in X_train_for_preprocessor_fit.columns],
    cat_cols=[c for c in cat_cols if c in X_train_for_preprocessor_fit.columns],
    scale_numeric=True
)

# Fit on TRAIN ONLY
pre_tree = get_fitted_preprocessor(pre_tree, X_train_for_preprocessor_fit)
pre_scaled = get_fitted_preprocessor(pre_scaled, X_train_for_preprocessor_fit)

# Transform combined Train+Val and Test
X_trva_tree = pre_tree.transform(X_trva_df)
X_trva_scaled = pre_scaled.transform(X_trva_df)

Xte_tree = pre_tree.transform(X_test_df)
Xte_scaled = pre_scaled.transform(X_test_df)

print("Shapes (tree):", X_trva_tree.shape, Xte_tree.shape)
print("Shapes (scaled):", X_trva_scaled.shape, Xte_scaled.shape)

Shapes (tree): (230785, 1002) (63259, 1002)
Shapes (scaled): (230785, 1002) (63259, 1002)


In [43]:
# Class weights computed on TRAIN ONLY (not train+val)
cw_bin = compute_class_weights(df_model[train_mask]["y_binary"].values.astype(int))
cw_mc  = compute_class_weights(df_model[train_mask]["y_multi"].values.astype(int))
print("Class weights (binary):", cw_bin)
print("Class weights (multiclass):", cw_mc)

Class weights (binary): {np.int64(0): np.float64(0.7821914880353373), np.int64(1): np.float64(1.38592324928204)}
Class weights (multiclass): {np.int64(0): np.float64(2.7249782824522213), np.int64(1): np.float64(0.3275192786718971), np.int64(2): np.float64(2.2676727253950224), np.int64(3): np.float64(7.205209187858901)}


In [44]:
# 10) Build models
models_bin = model_list_binary(class_weight_dict=cw_bin)
models_mc  = model_list_multiclass(class_weight_dict=cw_mc)
print("Binary models:", list(models_bin.keys()))
print("Multiclass models:", list(models_mc.keys()))


Binary models: ['LogReg', 'RF', 'MLP']
Multiclass models: ['LogReg', 'RF', 'MLP']


In [45]:
# 11) Train & validate — Binary
cv_store_bin = {}
fitted_bin = {}

for name, base_est in models_bin.items():
    pre = pre_scaled if name in ["LogReg", "MLP"] else pre_tree
    est = base_est

    cv_metrics, final_est = fit_with_cv(
        model_name=name, estimator=est,
        X=X_trva_tree if pre is pre_tree else X_trva_scaled,
        y=y_trva_bin_full, dates=dates_trva_full,
        task="binary", param_grid=None
    )
    cv_store_bin[name] = cv_metrics
    fitted_bin[name] = final_est

print("Finished Binary CV & refit on Train+Val.")

Finished Binary CV & refit on Train+Val.


In [57]:
# 12) Evaluate on Test — Binary
test_scores_bin = []
for name, est in fitted_bin.items():
    Xte = Xte_scaled if name in ["LogReg", "MLP", "LGBM"] else Xte_tree
    if hasattr(est, "predict_proba"):
        y_proba = est.predict_proba(Xte)[:, 1]
    elif hasattr(est, "decision_function"):
        scores = est.decision_function(Xte)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    else:
        y_proba = np.zeros_like(yte_bin, dtype=float)
    y_pred = est.predict(Xte)
    m = metric_summary_binary(yte_bin, y_pred, y_proba)
    m["model"] = name
    test_scores_bin.append(m)

df_bin_scores = pd.DataFrame(test_scores_bin).sort_values("f1", ascending=False)
df_bin_scores.to_csv(ARTIFACT_DIR / "binary_test_scores.csv", index=False)
print(df_bin_scores)

# Confidence intervals from CV
ci_rows = []
for name, metrics in cv_store_bin.items():
    for metric_name in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        if metric_name in metrics:
            mean, (low, high) = ci_from_folds(metrics[metric_name])
            ci_rows.append({"model": name, "metric": metric_name,
                            "mean": mean, "ci_low": low, "ci_high": high})
df_bin_ci = pd.DataFrame(ci_rows)
df_bin_ci.to_csv(ARTIFACT_DIR / "binary_cv_confidence_intervals.csv", index=False)

# Visuals
plot_cv_boxplots(cv_store_bin, task_name="binary")
plot_roc_pr_curves(fitted_bin, X_test=Xte_tree, y_test=yte_bin, task_name="binary", prefix="binary")

# Confusion matrices for top-N
for name in df_bin_scores["model"].head(4):
    est = fitted_bin[name]
    Xte = Xte_scaled if name in ["LogReg", "MLP", "LGBM"] else Xte_tree
    y_pred = est.predict(Xte)
    plot_confusion_and_reports(yte_bin, y_pred, labels=[0, 1],
                               title=f"{name} (Binary)", prefix=f"binary_{name}")

# Comparison bar chart (Binary)
plt.figure(figsize=(8, 5))
sns.barplot(x="model", y="f1", data=df_bin_scores, color="steelblue")
plt.title("Binary — F1 comparison on Test")
plt.ylabel("F1")
plt.tight_layout()
plt.savefig(FIG_DIR / "binary_f1_comparison.png", dpi=300)
plt.close()

   accuracy  precision    recall        f1   roc_auc   model
0  0.960875   0.944984  0.976269  0.960371  0.995395  LogReg
2  0.953335   0.952925  0.950877  0.951900  0.992620     MLP
1  0.944308   0.941379  0.944106  0.942741  0.989755      RF


In [47]:
# 13) Train & validate — Multi-class
cv_store_mc = {}
fitted_mc = {}
for name, base_est in models_mc.items():
    pre = pre_scaled if name in ["LogReg", "MLP"] else pre_tree
    est = base_est
    cv_metrics, final_est = fit_with_cv(
        model_name=name, estimator=est,
        X=X_trva_tree if pre is pre_tree else X_trva_scaled,
        y=y_trva_mc_full, dates=dates_trva_full,
        task="multiclass", param_grid=None
    )
    cv_store_mc[name] = cv_metrics
    fitted_mc[name] = final_est

print("Finished Multiclass CV & refit on Train+Val.")

Finished Multiclass CV & refit on Train+Val.


In [48]:
# 14) Evaluate on Test — Multi-class
test_scores_mc = []
for name, est in fitted_mc.items():
    Xte = Xte_scaled if name in ["LogReg", "MLP"] else Xte_tree
    y_pred = est.predict(Xte)
    m = metric_summary_multiclass(yte_mc, y_pred)
    m["model"] = name
    test_scores_mc.append(m)

df_mc_scores = pd.DataFrame(test_scores_mc).sort_values("f1_macro", ascending=False)
df_mc_scores.to_csv(ARTIFACT_DIR / "multiclass_test_scores.csv", index=False)
print(df_mc_scores)

# Confidence intervals from CV
ci_rows = []
for name, metrics in cv_store_mc.items():
    for metric_name in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]:
        if metric_name in metrics:
            mean, (low, high) = ci_from_folds(metrics[metric_name])
            ci_rows.append({"model": name, "metric": metric_name,
                            "mean": mean, "ci_low": low, "ci_high": high})
df_mc_ci = pd.DataFrame(ci_rows)
df_mc_ci.to_csv(ARTIFACT_DIR / "multiclass_cv_confidence_intervals.csv", index=False)

# Confusion matrices for top-N (MC)
for name in df_mc_scores["model"].head(4):
    est = fitted_mc[name]
    Xte = Xte_scaled if name in ["LogReg", "MLP"] else Xte_tree
    y_pred = est.predict(Xte)
    plot_confusion_and_reports(yte_mc, y_pred, labels=sorted(np.unique(yte_mc)),
                               title=f"{name} (Multi-class)", prefix=f"multiclass_{name}")

# Comparison bar chart (Macro-F1)
plt.figure(figsize=(8, 5))
sns.barplot(x="model", y="f1_macro", data=df_mc_scores, color="seagreen")
plt.title("Multi-class — Macro-F1 comparison on Test")
plt.ylabel("Macro-F1")
plt.tight_layout()
plt.savefig(FIG_DIR / "multiclass_macroF1_comparison.png", dpi=300)
plt.close()

   accuracy  precision_macro  recall_macro  f1_macro   model
2  0.943123         0.887747      0.840948  0.862642     MLP
0  0.905674         0.765647      0.915701  0.817189  LogReg
1  0.918336         0.914633      0.736825  0.804981      RF


In [49]:
# 15) Stacked Ensemble

# Binary stack
base_learners_bin = {k: v for k, v in models_bin.items() if k in ["RF", "LGBM", "XGB", "CAT"]}
if base_learners_bin:
    meta_lr_bin = LogisticRegression(max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
    stack_bin = stacked_ensemble(base_learners_bin, meta_lr_bin)
    stack_bin.fit(X_trva_tree, y_trva_bin_full)
    if hasattr(stack_bin, "predict_proba"):
        y_proba = stack_bin.predict_proba(Xte_tree)[:, 1]
    else:
        scores = stack_bin.decision_function(Xte_tree)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    y_pred = stack_bin.predict(Xte_tree)
    m_bin_stack = metric_summary_binary(yte_bin, y_pred, y_proba)
    m_bin_stack["model"] = "STACK_BIN"
    df_bin_scores = pd.concat([df_bin_scores, pd.DataFrame([m_bin_stack])], ignore_index=True)

# Multiclass stack
base_learners_mc = {k: v for k, v in models_mc.items() if k in ["RF", "LGBM", "XGB", "CAT"]}
if base_learners_mc:
    meta_lr_mc = LogisticRegression(max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
    stack_mc = stacked_ensemble(base_learners_mc, meta_lr_mc)
    stack_mc.fit(X_trva_tree, y_trva_mc_full)
    y_pred = stack_mc.predict(Xte_tree)
    m_mc_stack = metric_summary_multiclass(yte_mc, y_pred)
    m_mc_stack["model"] = "STACK_MC"
    df_mc_scores = pd.concat([df_mc_scores, pd.DataFrame([m_mc_stack])], ignore_index=True)

# Save with stacks included
df_bin_scores.sort_values("f1", ascending=False).to_csv(ARTIFACT_DIR / "binary_test_scores_with_stack.csv", index=False)
df_mc_scores.sort_values("f1_macro", ascending=False).to_csv(ARTIFACT_DIR / "multiclass_test_scores_with_stack.csv", index=False)

In [50]:
# 16) Leaderboard visualisations

def radar_plot(scores_df, score_col, title, filename):
    if scores_df.empty:
        return
    radar_metrics = [c for c in scores_df.columns if c not in ["model"] and scores_df[c].dtype != "O"]
    top = scores_df.sort_values(score_col, ascending=False).head(5)
    labels = list(top["model"])
    data = top[radar_metrics].values
    # Normalize to [0,1] per metric
    data_norm = (data - np.nanmin(data, axis=0)) / (np.nanmax(data, axis=0) - np.nanmin(data, axis=0) + 1e-9)

    angles = np.linspace(0, 2*np.pi, len(radar_metrics), endpoint=False)
    data_circ = np.concatenate([data_norm, data_norm[:, :1]], axis=1)
    angles_circ = np.concatenate([angles, angles[:1]])

    plt.figure(figsize=(7, 7))
    ax = plt.subplot(111, polar=True)
    for i, row in enumerate(data_circ):
        ax.plot(angles_circ, row, label=labels[i])
        ax.fill(angles_circ, row, alpha=0.1)
    ax.set_thetagrids(angles * 180/np.pi, radar_metrics)
    plt.title(title)
    plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300)
    plt.close()

# Heatmaps and radar
if not df_bin_scores.empty:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_bin_scores.set_index("model"), annot=True, fmt=".3f", cmap="YlGnBu")
    plt.title("Binary — Test leaderboard heatmap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "binary_leaderboard_heatmap.png", dpi=300)
    plt.close()
    radar_plot(df_bin_scores, score_col="f1", title="Binary — Model radar", filename="binary_radar.png")

if not df_mc_scores.empty:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_mc_scores.set_index("model"), annot=True, fmt=".3f", cmap="YlOrRd")
    plt.title("Multiclass — Test leaderboard heatmap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "multiclass_leaderboard_heatmap.png", dpi=300)
    plt.close()
    radar_plot(df_mc_scores, score_col="f1_macro", title="Multiclass — Model radar", filename="multiclass_radar.png")

In [51]:
# 17) Explainability with SHAP
if HAS_SHAP:
    feat_names_tree = get_feature_names(pre_tree)
    feat_names_scaled = get_feature_names(pre_scaled)

    best_bin = df_bin_scores.sort_values("f1", ascending=False).iloc[0]["model"] if not df_bin_scores.empty else None
    best_mc  = df_mc_scores.sort_values("f1_macro", ascending=False).iloc[0]["model"] if not df_mc_scores.empty else None

    n_shap = min(3000, Xte_tree.shape[0])
    idx_sample = np.random.RandomState(RANDOM_STATE).choice(np.arange(Xte_tree.shape[0]), size=n_shap, replace=False) if Xte_tree.shape[0] > 0 else np.array([], dtype=int)

    # Binary
    if best_bin is not None and best_bin in fitted_bin and n_shap > 0:
        model = fitted_bin[best_bin]
        X_shap = Xte_scaled[idx_sample] if best_bin in ["LogReg", "MLP"] else Xte_tree[idx_sample]
        names = feat_names_scaled if best_bin in ["LogReg", "MLP"] else feat_names_tree

        try:
            if best_bin in ["RF"] or (HAS_XGB and best_bin == "XGB") or (HAS_LGBM and best_bin == "LGBM") or (HAS_CAT and best_bin == "CAT"):
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_shap)
                shap_vals = shap_values[1] if isinstance(shap_values, list) and len(shap_values) == 2 else shap_values
            elif best_bin == "LogReg":
                explainer = shap.LinearExplainer(model, X_shap, feature_names=names)
                shap_vals = explainer.shap_values(X_shap)
            else:
                explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_shap, 200))
                shap_vals = explainer.shap_values(X_shap)[1]

            plt.figure()
            shap.summary_plot(shap_vals, features=X_shap, feature_names=names, show=False)
            plt.title(f"SHAP summary — {best_bin} (Binary)")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"shap_summary_binary_{best_bin}.png", dpi=300)
            plt.close()

            shap.plots.bar(shap.Explanation(values=shap_vals, data=X_shap, feature_names=names), show=False)
            plt.title(f"SHAP bar — {best_bin} (Binary)")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"shap_bar_binary_{best_bin}.png", dpi=300)
            plt.close()

            if isinstance(shap_vals, list):
                vals = shap_vals[0][0]
                base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
            else:
                vals = shap_vals[0]
                base = explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value[1]
            shap.plots.waterfall(shap.Explanation(values=vals, base_values=base, data=X_shap[0], feature_names=names), show=False)
            plt.title(f"SHAP waterfall — {best_bin} (Binary)")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"shap_waterfall_binary_{best_bin}.png", dpi=300)
            plt.close()
        except Exception as e:
            print("SHAP binary failed:", e)

    # Multiclass
    if best_mc is not None and best_mc in fitted_mc and n_shap > 0:
        model = fitted_mc[best_mc]
        X_shap = Xte_scaled[idx_sample] if best_mc in ["LogReg", "MLP"] else Xte_tree[idx_sample]
        names = feat_names_scaled if best_mc in ["LogReg", "MLP"] else feat_names_tree
        try:
            if best_mc in ["RF"] or (HAS_XGB and best_mc == "XGB") or (HAS_LGBM and best_mc == "LGBM") or (HAS_CAT and best_mc == "CAT"):
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_shap)  # list per class
                if isinstance(shap_values, list) and len(shap_values) > 1:
                    shap_abs_mean = np.mean([np.abs(sv) for sv in shap_values], axis=0)
                    plt.figure()
                    shap.summary_plot(shap_abs_mean, features=X_shap, feature_names=names, show=False)
                    plt.title(f"SHAP summary — {best_mc} (Multiclass)")
                    plt.tight_layout()
                    plt.savefig(FIG_DIR / f"shap_summary_multiclass_{best_mc}.png", dpi=300)
                    plt.close()
            elif best_mc == "LogReg":
                explainer = shap.LinearExplainer(model, X_shap, feature_names=names)
                shap_values = explainer.shap_values(X_shap)
                shap_abs_mean = np.mean([np.abs(sv) for sv in shap_values], axis=0)
                plt.figure()
                shap.summary_plot(shap_abs_mean, features=X_shap, feature_names=names, show=False)
                plt.title(f"SHAP summary — {best_mc} (Multiclass)")
                plt.tight_layout()
                plt.savefig(FIG_DIR / f"shap_summary_multiclass_{best_mc}.png", dpi=300)
                plt.close()
        except Exception as e:
            print("SHAP multiclass failed:", e)

In [52]:
# 18) Feature stability across folds — RandomForest example
if "RF" in models_bin:
    rf = models_bin["RF"]
    feat_importances = []
    for fold, (tr_idx, va_idx) in enumerate(rolling_time_series_folds(dates_trva_full), 1):
        rf_clone = RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=N_JOBS, class_weight=cw_bin
        )
        rf_clone.fit(X_trva_tree[tr_idx], y_trva_bin_full[tr_idx])
        if hasattr(rf_clone, "feature_importances_"):
            feat_importances.append(rf_clone.feature_importances_)
    if feat_importances:
        feat_importances = np.vstack(feat_importances)
        im_mean = feat_importances.mean(axis=0)
        im_std  = feat_importances.std(axis=0)
        names   = get_feature_names(pre_tree)
        top_idx = np.argsort(im_mean)[-20:][::-1]
        plt.figure(figsize=(10, 6))
        plt.barh(np.array(names)[top_idx][::-1], im_mean[top_idx][::-1], xerr=im_std[top_idx][::-1], color="teal")
        plt.title("Feature importance stability across folds — RF (Binary)")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "feature_stability_rf_binary.png", dpi=300)
        plt.close()

In [53]:
# 19) Partial dependence plots — Best binary model
try:
    if not df_bin_scores.empty:
        best_bin = df_bin_scores.sort_values("f1", ascending=False).iloc[0]["model"]
        est = fitted_bin[best_bin]
        Xte = Xte_scaled if best_bin in ["LogReg", "MLP"] else Xte_tree
        names = get_feature_names(pre_scaled if best_bin in ["LogReg", "MLP"] else pre_tree)
        top_features = names[:4] if len(names) >= 4 else names
        fig, ax = plt.subplots(2, 2, figsize=(10, 8))
        PartialDependenceDisplay.from_estimator(est, Xte, features=top_features[:4], feature_names=names, ax=ax.ravel())
        plt.suptitle(f"Partial dependence — {best_bin} (Binary)")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "pdp_best_binary.png", dpi=300)
        plt.close()
except Exception as e:
    print("PDP generation failed:", e)

In [54]:
# 20) Save a compact textual summary
with open(ARTIFACT_DIR / "summary.txt", "w") as f:
    f.write("Binary test scores (sorted by F1):\n")
    f.write(df_bin_scores.sort_values("f1", ascending=False).to_string(index=False))
    f.write("\n\nMulticlass test scores (sorted by Macro-F1):\n")
    f.write(df_mc_scores.sort_values("f1_macro", ascending=False).to_string(index=False))
    f.write("\n\nBinary CV CIs:\n")
    f.write((ARTIFACT_DIR / "binary_cv_confidence_intervals.csv").as_posix())
    f.write("\nMulticlass CV CIs:\n")
    f.write((ARTIFACT_DIR / "multiclass_cv_confidence_intervals.csv").as_posix())

print("Pipeline complete.")
print("Artifacts →", ARTIFACT_DIR.resolve())
print("Figures   →", FIG_DIR.resolve())

Pipeline complete.
Artifacts → /content/artifacts
Figures   → /content/figs


In [58]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


In [59]:
# ======================================================
# SHAP Explainability (Beeswarm + Bar + Dependence + Local)
# for Binary Ablation Scenarios: LogReg | MLP | RF | STACK_BIN
# ======================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional imports guarded in your pipeline
HAS_XGB = False
HAS_LGBM = False
HAS_CAT = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    pass

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    pass

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    pass

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False


# ------------------ Helpers: data prep per scenario ------------------
def _scenario_feature_lists(df, features_all, drop_cols):
    """Return kept features, and split into numeric/categorical."""
    feat_keep = [c for c in features_all if c not in drop_cols]
    num_keep = [c for c in feat_keep if (c in df.columns and pd.api.types.is_numeric_dtype(df[c]))]
    cat_keep = [c for c in feat_keep if c not in num_keep]
    return feat_keep, num_keep, cat_keep


def _fit_preprocessors_for_scenario(df, train_mask, feat_keep, num_keep, cat_keep):
    """Fit preprocessors (train-only leakage-safe) and return fitted ones."""
    X_train_fit = df.loc[train_mask, feat_keep].copy()
    pre_tree = make_preprocessor(num_cols=num_keep, cat_cols=cat_keep, scale_numeric=False)
    pre_scaled = make_preprocessor(num_cols=num_keep, cat_cols=cat_keep, scale_numeric=True)
    pre_tree = get_fitted_preprocessor(pre_tree, X_train_fit)
    pre_scaled = get_fitted_preprocessor(pre_scaled, X_train_fit)
    return pre_tree, pre_scaled


def _transform_partitions(df, feat_keep, pre_tree, pre_scaled, train_mask, val_mask, test_mask):
    """Return transformed X (train+val, test), y (train+val, test) with both preprocessors."""
    # Train+Val chunk (sorted chronologically and 0-indexed)
    trval = df.loc[train_mask | val_mask, ["y_binary", "_date"] + feat_keep].sort_values("_date").reset_index(drop=True)
    X_trva_df = trval[feat_keep]
    y_trva = trval["y_binary"].values.astype(int)

    # Test chunk
    test = df.loc[test_mask, ["y_binary", "_date"] + feat_keep]
    X_test_df = test[feat_keep]
    y_test = test["y_binary"].values.astype(int)

    # Transform
    X_trva_tree = pre_tree.transform(X_trva_df)
    X_trva_scaled = pre_scaled.transform(X_trva_df)
    Xte_tree     = pre_tree.transform(X_test_df)
    Xte_scaled   = pre_scaled.transform(X_test_df)

    return X_trva_tree, X_trva_scaled, Xte_tree, Xte_scaled, y_trva, y_test


def _build_models_bin(cw_bin):
    """Return the three base models + stack for binary classification."""
    base = model_list_binary(class_weight_dict=cw_bin)
    models = {}
    # Only keep what we need
    for k in ["LogReg", "MLP", "RF"]:
        if k in base:
            models[k] = base[k]
    # Build stack (use whatever base learners are available)
    base_learners = {}
    for k in ["RF", "LGBM", "XGB", "CAT"]:
        if k in base:
            base_learners[k] = base[k]
    if len(base_learners) > 0:
        meta_lr = LogisticRegression(max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
        models["STACK_BIN"] = stacked_ensemble(base_learners, meta_lr)
    return models


# ------------------ SHAP explainers per model type ------------------
def _make_shap_explainer(model_name, model, X_background, feature_names):
    """
    Return a SHAP explainer appropriate to the model.
    - Trees: TreeExplainer
    - LogReg: LinearExplainer
    - Others: KernelExplainer with sampled background
    """
    if not HAS_SHAP:
        return None, None

    # Heuristics for model family
    is_tree = isinstance(model, (RandomForestClassifier,)) \
              or (HAS_XGB and isinstance(model, XGBClassifier)) \
              or (HAS_LGBM and isinstance(model, LGBMClassifier)) \
              or (HAS_CAT and isinstance(model, CatBoostClassifier)) \
              or ("StackingClassifier" in model.__class__.__name__)  # STACK_BIN often tree-based base

    try:
        if is_tree and model_name != "LogReg" and model_name != "MLP":
            explainer = shap.TreeExplainer(model)
            expected_value = explainer.expected_value
            return explainer, expected_value
        elif model_name == "LogReg":
            # LinearExplainer expects background to estimate link behavior
            explainer = shap.LinearExplainer(model, X_background, feature_names=feature_names)
            return explainer, explainer.expected_value
        else:
            # Kernel for MLP or unknown, with a small background
            bg = shap.sample(X_background, min(200, X_background.shape[0]))
            explainer = shap.KernelExplainer(model.predict_proba, bg)
            return explainer, explainer.expected_value
    except Exception as e:
        print(f"[WARN] SHAP explainer creation failed for {model_name}: {e}")
        return None, None


def _compute_shap_values_binary(model_name, model, explainer, X_sample):
    """Compute SHAP values -> class-1 attributions for binary."""
    try:
        shap_vals = explainer.shap_values(X_sample)
        # Tree/Linear may return list [class0, class1]
        if isinstance(shap_vals, list):
            if len(shap_vals) == 2:
                return shap_vals[1]
            # If shape is for multiclass but 2 classes, still index 1
            return shap_vals[-1]
        return shap_vals
    except Exception as e:
        print(f"[WARN] SHAP value computation failed for {model_name}: {e}")
        return None


# ------------------ Plotting utilities ------------------
def _ensure_dir(path_obj):
    path_obj.mkdir(parents=True, exist_ok=True)


def plot_shap_beeswarm(shap_values, X_sample, feature_names, title, out_path):
    """Global beeswarm (aka 'Beesham') plot."""
    plt.figure()
    shap.summary_plot(shap_values, features=X_sample, feature_names=feature_names, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def plot_shap_bar(shap_values, X_sample, feature_names, title, out_path):
    """Mean |SHAP| bar chart (top-20 by default from shap)."""
    exp = shap.Explanation(values=shap_values, data=X_sample, feature_names=feature_names)
    shap.plots.bar(exp, show=False, max_display=20)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def plot_shap_dependence_topk(shap_values, X_sample, feature_names, topk=3, prefix_title="", prefix_file=""):
    """
    SHAP dependence plots for the top-k features by mean|SHAP|.
    Saves individual PNGs.
    """
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    order = np.argsort(mean_abs)[::-1][:topk]
    for i, j in enumerate(order, 1):
        fname = feature_names[j]
        try:
            plt.figure()
            shap.dependence_plot(
                ind=j,
                shap_values=shap_values,
                features=X_sample,
                feature_names=feature_names,
                interaction_index="auto",
                show=False
            )
            plt.title(f"{prefix_title} — Dependence: {fname}")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"{prefix_file}_dependence_{i}_{fname[:40]}.png", dpi=300)
            plt.close()
        except Exception as e:
            print(f"[WARN] Dependence plot failed for {fname}: {e}")


def plot_shap_waterfall_single(explainer, shap_values, X_sample, feature_names, instance_index, title, out_path):
    """
    Local (single prediction) SHAP waterfall plot.
    Picks instance_index from X_sample.
    """
    try:
        vals = shap_values[instance_index]
        # Handle expected_value shape differences
        base_val = explainer.expected_value
        if isinstance(base_val, (list, tuple, np.ndarray)):
            base_val = base_val[1] if len(np.atleast_1d(base_val)) > 1 else np.atleast_1d(base_val)[0]

        exp = shap.Explanation(values=vals, base_values=base_val, data=X_sample[instance_index], feature_names=feature_names)
        shap.plots.waterfall(exp, show=False, max_display=20)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(out_path, dpi=300)
        plt.close()
    except Exception as e:
        print(f"[WARN] Waterfall plot failed: {e}")


def shap_stability_heatmap(shap_dict, feature_names_union, title, out_path):
    """
    Compare mean|SHAP| across scenarios for a model.
    shap_dict: {scenario_label: (shap_values, feature_names)}
    feature_names_union: common list to display in fixed order
    """
    mats = []
    rows = []
    for scen, (sv, fnames) in shap_dict.items():
        # Map to union order
        name_to_idx = {n: i for i, n in enumerate(fnames)}
        aligned = []
        for f in feature_names_union:
            if f in name_to_idx:
                aligned.append(np.mean(np.abs(sv[:, name_to_idx[f]])))
            else:
                aligned.append(np.nan)
        mats.append(aligned)
        rows.append(scen)
    M = np.array(mats)
    dfM = pd.DataFrame(M, index=rows, columns=feature_names_union)
    plt.figure(figsize=(min(16, 1.2*len(feature_names_union)), max(3.5, 0.5*len(rows))))
    sns.heatmap(dfM, cmap="magma", annot=False)
    plt.title(title)
    plt.xlabel("Feature")
    plt.ylabel("Scenario")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


# ------------------ Main entry: compute and plot SHAP per scenario/model ------------------
def run_shap_explainability_for_ablation(
    df,
    avg_col, pct_cols, num_cols, cat_cols,
    train_mask, val_mask, test_mask,
    dates_trva_full,
    cw_bin,
    scenarios,            # list of dicts: [{"name": "ALL", "drop": []}, {"name":"NO_LAGS", "drop":[...]}]
    models=("LogReg", "MLP", "RF", "STACK_BIN"),
    max_test_samples=2000 # subsample test for speed if large
):
    """
    For each scenario and model:
      - Build leakage-safe preprocessor
      - Fit (train+val) the model (uses your CV function to pick best fold config)
      - Compute SHAP on test subsample
      - Save beeswarm, bar, dependence (top-3), and one local waterfall

    Also produces per-model stability heatmaps across scenarios based on mean|SHAP|.
    """
    if not HAS_SHAP:
        print("SHAP not available. Please `pip install shap` and rerun.")
        return

    _ensure_dir(FIG_DIR / "ablation_shap")

    features_all = num_cols + cat_cols
    models_all = _build_models_bin(cw_bin)

    # Filter to only requested models (present)
    models_to_use = {k: v for k, v in models_all.items() if k in models}

    # Store mean|SHAP| across scenarios per model for stability plots
    per_model_scenario_shap = {m: {} for m in models_to_use.keys()}

    for sc in scenarios:
        sc_name = sc["name"]
        drop_cols = sc.get("drop", [])
        print(f"\n[SHAP] Scenario: {sc_name}")

        # 1) Feature selection for scenario
        feat_keep, num_keep, cat_keep = _scenario_feature_lists(df, features_all, drop_cols)
        if len(feat_keep) == 0:
            print(f"[skip] No features left after drop for scenario {sc_name}")
            continue

        # 2) Preprocessors fit on TRAIN only
        pre_tree, pre_scaled = _fit_preprocessors_for_scenario(df, train_mask, feat_keep, num_keep, cat_keep)

        # 3) Transform partitions
        X_trva_tree, X_trva_scaled, Xte_tree, Xte_scaled, y_trva, y_test = _transform_partitions(
            df, feat_keep, pre_tree, pre_scaled, train_mask, val_mask, test_mask
        )

        # Subsample test for SHAP speed if needed
        if Xte_tree.shape[0] > max_test_samples:
            rng = np.random.RandomState(RANDOM_STATE)
            sel = rng.choice(np.arange(Xte_tree.shape[0]), size=max_test_samples, replace=False)
            Xte_tree_sub   = Xte_tree[sel]
            Xte_scaled_sub = Xte_scaled[sel]
            y_test_sub     = y_test[sel]
        else:
            Xte_tree_sub, Xte_scaled_sub, y_test_sub = Xte_tree, Xte_scaled, y_test

        # Feature names for plots
        names_tree   = get_feature_names(pre_tree)
        names_scaled = get_feature_names(pre_scaled)

        # 4) Train each model for this scenario (using rolling CV) and compute SHAP
        for mdl_name, base_est in models_to_use.items():
            print(f"  - Model: {mdl_name}")

            # Choose representation
            if mdl_name in ["LogReg", "MLP"]:
                X_trva_use = X_trva_scaled
                Xte_use    = Xte_scaled_sub
                names_use  = names_scaled
            else:
                X_trva_use = X_trva_tree
                Xte_use    = Xte_tree_sub
                names_use  = names_tree

            # Fit with rolling CV and refit on full tr+val
            cv_metrics, final_est = fit_with_cv(
                model_name=mdl_name, estimator=base_est,
                X=X_trva_use, y=y_trva, dates=dates_trva_full,
                task="binary", param_grid=None
            )

            # 5) SHAP Explainer
            explainer, expected_value = _make_shap_explainer(
                mdl_name, final_est, X_trva_use, names_use
            )
            if explainer is None:
                print(f"    [skip] No SHAP explainer for {mdl_name} in {sc_name}")
                continue

            # 6) Compute SHAP for class-1 (delayed)
            shap_vals = _compute_shap_values_binary(mdl_name, final_est, explainer, Xte_use)
            if shap_vals is None:
                continue

            # 7) Global beeswarm + bar
            plot_shap_beeswarm(
                shap_values=shap_vals,
                X_sample=Xte_use,
                feature_names=names_use,
                title=f"{mdl_name} — {sc_name} — SHAP beeswarm (Binary)",
                out_path=FIG_DIR / "ablation_shap" / f"shap_beeswarm_{mdl_name}_{sc_name}.png"
            )
            plot_shap_bar(
                shap_values=shap_vals,
                X_sample=Xte_use,
                feature_names=names_use,
                title=f"{mdl_name} — {sc_name} — Mean |SHAP| (Top-20)",
                out_path=FIG_DIR / "ablation_shap" / f"shap_bar_{mdl_name}_{sc_name}.png"
            )

            # 8) Dependence plots for top-3 features
            plot_shap_dependence_topk(
                shap_values=shap_vals,
                X_sample=Xte_use,
                feature_names=names_use,
                topk=3,
                prefix_title=f"{mdl_name} — {sc_name}",
                prefix_file=f"ablation_shap/shap_{mdl_name}_{sc_name}"
            )

            # 9) Local waterfall for one representative delayed case (if any)
            try:
                # Choose an instance with y=1 if present; else index 0
                idx = int(np.where(y_test_sub == 1)[0][0]) if np.any(y_test_sub == 1) else 0
                plot_shap_waterfall_single(
                    explainer=explainer,
                    shap_values=shap_vals,
                    X_sample=Xte_use,
                    feature_names=names_use,
                    instance_index=idx,
                    title=f"{mdl_name} — {sc_name} — Local Waterfall (Delayed example)",
                    out_path=FIG_DIR / "ablation_shap" / f"shap_waterfall_{mdl_name}_{sc_name}.png"
                )
            except Exception as e:
                print(f"[WARN] Could not produce waterfall for {mdl_name}/{sc_name}: {e}")

            # Store for stability heatmap later
            per_model_scenario_shap[mdl_name][sc_name] = (shap_vals, names_use)

    # ---------------- Stability Heatmaps per model ----------------
    for mdl_name, scen_map in per_model_scenario_shap.items():
        if len(scen_map) <= 1:
            continue
        # Build union of top features across scenarios (by mean|SHAP|)
        top_union = set()
        for sc_name, (sv, fnames) in scen_map.items():
            mean_abs = np.mean(np.abs(sv), axis=0)
            top_idx = np.argsort(mean_abs)[::-1][:15]
            top_union.update([fnames[i] for i in top_idx])
        top_union = list(top_union)

        shap_stability_heatmap(
            shap_dict=scen_map,
            feature_names_union=top_union,
            title=f"SHAP Stability across scenarios — {mdl_name}",
            out_path=FIG_DIR / "ablation_shap" / f"shap_stability_{mdl_name}.png"
        )

    print("SHAP explainability done. See:", (FIG_DIR / "ablation_shap").resolve())

In [60]:
def _scenario_feature_lists(df, features_all, drop_cols):
    """Return kept features, and split into numeric/categorical."""
    feat_keep = [c for c in features_all if c not in drop_cols]
    num_keep = [c for c in feat_keep if (c in df.columns and pd.api.types.is_numeric_dtype(df[c]))]
    cat_keep = [c for c in feat_keep if c not in num_keep]
    return feat_keep, num_keep, cat_keep


In [61]:
def _fit_preprocessors_for_scenario(df, train_mask, feat_keep, num_keep, cat_keep):
    """Fit preprocessors (train-only leakage-safe) and return fitted ones."""
    X_train_fit = df.loc[train_mask, feat_keep].copy()
    pre_tree = make_preprocessor(num_cols=num_keep, cat_cols=cat_keep, scale_numeric=False)
    pre_scaled = make_preprocessor(num_cols=num_keep, cat_cols=cat_keep, scale_numeric=True)
    pre_tree = get_fitted_preprocessor(pre_tree, X_train_fit)
    pre_scaled = get_fitted_preprocessor(pre_scaled, X_train_fit)
    return pre_tree, pre_scaled


In [62]:
def _transform_partitions(df, feat_keep, pre_tree, pre_scaled, train_mask, val_mask, test_mask):
    """Return transformed X (train+val, test), y (train+val, test) with both preprocessors."""
    # Train+Val chunk (sorted chronologically and 0-indexed)
    trval = df.loc[train_mask | val_mask, ["y_binary", "_date"] + feat_keep].sort_values("_date").reset_index(drop=True)
    X_trva_df = trval[feat_keep]
    y_trva = trval["y_binary"].values.astype(int)

    # Test chunk
    test = df.loc[test_mask, ["y_binary", "_date"] + feat_keep]
    X_test_df = test[feat_keep]
    y_test = test["y_binary"].values.astype(int)

    # Transform
    X_trva_tree = pre_tree.transform(X_trva_df)
    X_trva_scaled = pre_scaled.transform(X_trva_df)
    Xte_tree     = pre_tree.transform(X_test_df)
    Xte_scaled   = pre_scaled.transform(X_test_df)

    return X_trva_tree, X_trva_scaled, Xte_tree, Xte_scaled, y_trva, y_test


In [63]:
def _build_models_bin(cw_bin):
    """Return the three base models + stack for binary classification."""
    base = model_list_binary(class_weight_dict=cw_bin)
    models = {}
    # Only keep what we need
    for k in ["LogReg", "MLP", "RF"]:
        if k in base:
            models[k] = base[k]
    # Build stack (use whatever base learners are available)
    base_learners = {}
    for k in ["RF", "LGBM", "XGB", "CAT"]:
        if k in base:
            base_learners[k] = base[k]
    if len(base_learners) > 0:
        meta_lr = LogisticRegression(max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
        models["STACK_BIN"] = stacked_ensemble(base_learners, meta_lr)
    return models


In [64]:
# ------------------ SHAP explainers per model type ------------------
def _make_shap_explainer(model_name, model, X_background, feature_names):
    """
    Return a SHAP explainer appropriate to the model.
    - Trees: TreeExplainer
    - LogReg: LinearExplainer
    - Others: KernelExplainer with sampled background
    """
    if not HAS_SHAP:
        return None, None

    # Heuristics for model family
    is_tree = isinstance(model, (RandomForestClassifier,)) \
              or (HAS_XGB and isinstance(model, XGBClassifier)) \
              or (HAS_LGBM and isinstance(model, LGBMClassifier)) \
              or (HAS_CAT and isinstance(model, CatBoostClassifier)) \
              or ("StackingClassifier" in model.__class__.__name__)  # STACK_BIN often tree-based base

    try:
        if is_tree and model_name != "LogReg" and model_name != "MLP":
            explainer = shap.TreeExplainer(model)
            expected_value = explainer.expected_value
            return explainer, expected_value
        elif model_name == "LogReg":
            # LinearExplainer expects background to estimate link behavior
            explainer = shap.LinearExplainer(model, X_background, feature_names=feature_names)
            return explainer, explainer.expected_value
        else:
            # Kernel for MLP or unknown, with a small background
            bg = shap.sample(X_background, min(200, X_background.shape[0]))
            explainer = shap.KernelExplainer(model.predict_proba, bg)
            return explainer, explainer.expected_value
    except Exception as e:
        print(f"[WARN] SHAP explainer creation failed for {model_name}: {e}")
        return None, None



In [65]:
def _compute_shap_values_binary(model_name, model, explainer, X_sample):
    """Compute SHAP values -> class-1 attributions for binary."""
    try:
        shap_vals = explainer.shap_values(X_sample)
        # Tree/Linear may return list [class0, class1]
        if isinstance(shap_vals, list):
            if len(shap_vals) == 2:
                return shap_vals[1]
            # If shape is for multiclass but 2 classes, still index 1
            return shap_vals[-1]
        return shap_vals
    except Exception as e:
        print(f"[WARN] SHAP value computation failed for {model_name}: {e}")
        return None


In [66]:
# ------------------ Plotting utilities ------------------
def _ensure_dir(path_obj):
    path_obj.mkdir(parents=True, exist_ok=True)

def plot_shap_beeswarm(shap_values, X_sample, feature_names, title, out_path):
    """Global beeswarm (aka 'Beesham') plot."""
    plt.figure()
    shap.summary_plot(shap_values, features=X_sample, feature_names=feature_names, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


In [67]:
def plot_shap_bar(shap_values, X_sample, feature_names, title, out_path):
    """Mean |SHAP| bar chart (top-20 by default from shap)."""
    exp = shap.Explanation(values=shap_values, data=X_sample, feature_names=feature_names)
    shap.plots.bar(exp, show=False, max_display=20)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


In [68]:
def plot_shap_dependence_topk(shap_values, X_sample, feature_names, topk=3, prefix_title="", prefix_file=""):
    """
    SHAP dependence plots for the top-k features by mean|SHAP|.
    Saves individual PNGs.
    """
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    order = np.argsort(mean_abs)[::-1][:topk]
    for i, j in enumerate(order, 1):
        fname = feature_names[j]
        try:
            plt.figure()
            shap.dependence_plot(
                ind=j,
                shap_values=shap_values,
                features=X_sample,
                feature_names=feature_names,
                interaction_index="auto",
                show=False
            )
            plt.title(f"{prefix_title} — Dependence: {fname}")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"{prefix_file}_dependence_{i}_{fname[:40]}.png", dpi=300)
            plt.close()
        except Exception as e:
            print(f"[WARN] Dependence plot failed for {fname}: {e}")


In [69]:
def plot_shap_waterfall_single(explainer, shap_values, X_sample, feature_names, instance_index, title, out_path):
    """
    Local (single prediction) SHAP waterfall plot.
    Picks instance_index from X_sample.
    """
    try:
        vals = shap_values[instance_index]
        # Handle expected_value shape differences
        base_val = explainer.expected_value
        if isinstance(base_val, (list, tuple, np.ndarray)):
            base_val = base_val[1] if len(np.atleast_1d(base_val)) > 1 else np.atleast_1d(base_val)[0]

        exp = shap.Explanation(values=vals, base_values=base_val, data=X_sample[instance_index], feature_names=feature_names)
        shap.plots.waterfall(exp, show=False, max_display=20)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(out_path, dpi=300)
        plt.close()
    except Exception as e:
        print(f"[WARN] Waterfall plot failed: {e}")


In [70]:
def shap_stability_heatmap(shap_dict, feature_names_union, title, out_path):
    """
    Compare mean|SHAP| across scenarios for a model.
    shap_dict: {scenario_label: (shap_values, feature_names)}
    feature_names_union: common list to display in fixed order
    """
    mats = []
    rows = []
    for scen, (sv, fnames) in shap_dict.items():
        # Map to union order
        name_to_idx = {n: i for i, n in enumerate(fnames)}
        aligned = []
        for f in feature_names_union:
            if f in name_to_idx:
                aligned.append(np.mean(np.abs(sv[:, name_to_idx[f]])))
            else:
                aligned.append(np.nan)
        mats.append(aligned)
        rows.append(scen)
    M = np.array(mats)
    dfM = pd.DataFrame(M, index=rows, columns=feature_names_union)
    plt.figure(figsize=(min(16, 1.2*len(feature_names_union)), max(3.5, 0.5*len(rows))))
    sns.heatmap(dfM, cmap="magma", annot=False)
    plt.title(title)
    plt.xlabel("Feature")
    plt.ylabel("Scenario")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


In [73]:
# ------------------ Main entry: compute and plot SHAP per scenario/model ------------------
def run_shap_explainability_for_ablation(
    df,
    avg_col, pct_cols, num_cols, cat_cols,
    train_mask, val_mask, test_mask,
    dates_trva_full,
    cw_bin,
    scenarios,            # list of dicts: [{"name": "ALL", "drop": []}, {"name":"NO_LAGS", "drop":[...]}]
    models=("LogReg", "MLP", "RF", "STACK_BIN"),
    max_test_samples=2000 # subsample test for speed if large
):
    """
    For each scenario and model:
      - Build leakage-safe preprocessor
      - Fit (train+val) the model (uses your CV function to pick best fold config)
      - Compute SHAP on test subsample
      - Save beeswarm, bar, dependence (top-3), and one local waterfall

    Also produces per-model stability heatmaps across scenarios based on mean|SHAP|.
    """
    if not HAS_SHAP:
        print("SHAP not available. Please `pip install shap` and rerun.")
        return

    _ensure_dir(FIG_DIR / "ablation_shap")

    features_all = num_cols + cat_cols
    models_all = _build_models_bin(cw_bin)

    # Filter to only requested models (present)
    models_to_use = {k: v for k, v in models_all.items() if k in models}

    # Store mean|SHAP| across scenarios per model for stability plots
    per_model_scenario_shap = {m: {} for m in models_to_use.keys()}

    for sc in scenarios:
        sc_name = sc["name"]
        drop_cols = sc.get("drop", [])
        print(f"\n[SHAP] Scenario: {sc_name}")

        # 1) Feature selection for scenario
        feat_keep, num_keep, cat_keep = _scenario_feature_lists(df, features_all, drop_cols)
        if len(feat_keep) == 0:
            print(f"[skip] No features left after drop for scenario {sc_name}")
            continue

        # 2) Preprocessors fit on TRAIN only
        pre_tree, pre_scaled = _fit_preprocessors_for_scenario(df, train_mask, feat_keep, num_keep, cat_keep)

        # 3) Transform partitions
        X_trva_tree, X_trva_scaled, Xte_tree, Xte_scaled, y_trva, y_test = _transform_partitions(
            df, feat_keep, pre_tree, pre_scaled, train_mask, val_mask, test_mask
        )

        # Subsample test for SHAP speed if needed
        if Xte_tree.shape[0] > max_test_samples:
            rng = np.random.RandomState(RANDOM_STATE)
            sel = rng.choice(np.arange(Xte_tree.shape[0]), size=max_test_samples, replace=False)
            Xte_tree_sub   = Xte_tree[sel]
            Xte_scaled_sub = Xte_scaled[sel]
            y_test_sub     = y_test[sel]
        else:
            Xte_tree_sub, Xte_scaled_sub, y_test_sub = Xte_tree, Xte_scaled, y_test

        # Feature names for plots
        names_tree   = get_feature_names(pre_tree)
        names_scaled = get_feature_names(pre_scaled)

        # 4) Train each model for this scenario (using rolling CV) and compute SHAP
        for mdl_name, base_est in models_to_use.items():
            print(f"  - Model: {mdl_name}")

            # Choose representation
            if mdl_name in ["LogReg", "MLP"]:
                X_trva_use = X_trva_scaled
                Xte_use    = Xte_scaled_sub
                names_use  = names_scaled
            else:
                X_trva_use = X_trva_tree
                Xte_use    = Xte_tree_sub
                names_use  = names_tree

            # Fit with rolling CV and refit on full tr+val
            cv_metrics, final_est = fit_with_cv(
                model_name=mdl_name, estimator=base_est,
                X=X_trva_use, y=y_trva, dates=dates_trva_full,
                task="binary", param_grid=None
            )

            # 5) SHAP Explainer
            explainer, expected_value = _make_shap_explainer(
                mdl_name, final_est, X_trva_use, names_use
            )
            if explainer is None:
                print(f"    [skip] No SHAP explainer for {mdl_name} in {sc_name}")
                continue

            # 6) Compute SHAP for class-1 (delayed)
            shap_vals = _compute_shap_values_binary(mdl_name, final_est, explainer, Xte_use)
            if shap_vals is None:
                continue

            # 7) Global beeswarm + bar
            plot_shap_beeswarm(
                shap_values=shap_vals,
                X_sample=Xte_use,
                feature_names=names_use,
                title=f"{mdl_name} — {sc_name} — SHAP beeswarm (Binary)",
                out_path=FIG_DIR / "ablation_shap" / f"shap_beeswarm_{mdl_name}_{sc_name}.png"
            )
            plot_shap_bar(
                shap_values=shap_vals,
                X_sample=Xte_use,
                feature_names=names_use,
                title=f"{mdl_name} — {sc_name} — Mean |SHAP| (Top-20)",
                out_path=FIG_DIR / "ablation_shap" / f"shap_bar_{mdl_name}_{sc_name}.png"
            )

            # 8) Dependence plots for top-3 features
            plot_shap_dependence_topk(
                shap_values=shap_vals,
                X_sample=Xte_use,
                feature_names=names_use,
                topk=3,
                prefix_title=f"{mdl_name} — {sc_name}",
                prefix_file=f"ablation_shap/shap_{mdl_name}_{sc_name}"
            )

            # 9) Local waterfall for one representative delayed case (if any)
            try:
                # Choose an instance with y=1 if present; else index 0
                idx = int(np.where(y_test_sub == 1)[0][0]) if np.any(y_test_sub == 1) else 0
                plot_shap_waterfall_single(
                    explainer=explainer,
                    shap_values=shap_vals,
                    X_sample=Xte_use,
                    feature_names=names_use,
                    instance_index=idx,
                    title=f"{mdl_name} — {sc_name} — Local Waterfall (Delayed example)",
                    out_path=FIG_DIR / "ablation_shap" / f"shap_waterfall_{mdl_name}_{sc_name}.png"
                )
            except Exception as e:
                print(f"[WARN] Could not produce waterfall for {mdl_name}/{sc_name}: {e}")

            # Store for stability heatmap later
            per_model_scenario_shap[mdl_name][sc_name] = (shap_vals, names_use)

    # ---------------- Stability Heatmaps per model ----------------
    for mdl_name, scen_map in per_model_scenario_shap.items():
        if len(scen_map) <= 1:
            continue
        # Build union of top features across scenarios (by mean|SHAP|)
        top_union = set()
        for sc_name, (sv, fnames) in scen_map.items():
            mean_abs = np.mean(np.abs(sv), axis=0)
            top_idx = np.argsort(mean_abs)[::-1][:15]
            top_union.update([fnames[i] for i in top_idx])
        top_union = list(top_union)

        shap_stability_heatmap(
            shap_dict=scen_map,
            feature_names_union=top_union,
            title=f"SHAP Stability across scenarios — {mdl_name}",
            out_path=FIG_DIR / "ablation_shap" / f"shap_stability_{mdl_name}.png"
        )

    print("SHAP explainability done. See:", (FIG_DIR / "ablation_shap").resolve())



In [75]:
def detect_feature_blocks(df, avg_col, pct_cols, num_cols, cat_cols):
    blocks = {}

    # Temporal block
    temporal_cols = [c for c in ["year", "month", "season", "time_index"] if c in df.columns]
    blocks["TEMPORAL"] = temporal_cols

    # Percent bins (the nine punctuality/late-share columns from the source)
    blocks["PERCENT_BINS"] = [c for c in pct_cols if c in df.columns]

    # Lag features (created earlier; names contain "_lag")
    blocks["LAGS"] = [c for c in df.columns if "_lag" in c and c not in [avg_col]]

    # Rolling features (created earlier; names contain "_roll")
    blocks["ROLLING"] = [c for c in df.columns if "_roll" in c]

    # Derived shares / proxies we added
    derived = [c for c in ["pct_over_15", "pct_on_time", "traffic_intensity_proxy"] if c in df.columns]
    blocks["DERIVED"] = derived

    # Categorical IDs (one-hot encoded later)
    blocks["CATEGORICAL"] = [c for c in cat_cols if c in df.columns]

    # Numeric base (catch-all numeric not in the above, excluding targets & date)
    exclude = set([avg_col, "y_binary", "y_multi", "_date", "group_id"] +
                  blocks["TEMPORAL"] + blocks["PERCENT_BINS"] + blocks["LAGS"] +
                  blocks["ROLLING"] + blocks["DERIVED"] + blocks["CATEGORICAL"])
    numeric_base = [c for c in num_cols if c not in exclude]
    blocks["NUMERIC_BASE"] = numeric_base  # often empty (ok)

    return blocks


In [76]:
def build_ablation_scenarios(blocks, full_feature_list):
    scenarios = []
    # Baseline: ALL features (no drop)
    scenarios.append({"name": "ALL", "drop": []})

    # Drop one block at a time
    for block_name, cols in blocks.items():
        if len(cols) == 0:
            continue
        scenarios.append({"name": f"NO_{block_name}", "drop": cols})

    # Optional: compound ablations (examples)
    # scenarios.append({"name": "NO_LAGS_NO_ROLLING", "drop": blocks["LAGS"] + blocks["ROLLING"]})

    # Ensure each scenario retains at least one predictor
    valid = []
    for sc in scenarios:
        keep = [c for c in full_feature_list if c not in sc["drop"]]
        if len(keep) > 0:
            valid.append(sc)
    return valid


In [77]:
def run_one_ablation_binary(
    model_name,
    base_estimator,
    features_all,
    drops,
    df_model,
    train_mask, val_mask, test_mask,
    dates_trva_full,
    cw_bin,
    pre_tree_factory,
    pre_scaled_factory
):
    """
    - features_all: list of candidate features (num+cat)
    - drops: list of columns to drop in the ablation
    - df_model: df[features + ["y_binary","_date"]]
    - pre_*_factory: functions that build (not fit) preprocessors with supplied (num, cat)
    """
    # 1) Choose features to keep
    feat_keep = [c for c in features_all if c not in drops]
    if len(feat_keep) == 0:
        return None  # skip

    # Separate partitions
    trval = df_model[train_mask | val_mask].sort_values("_date").reset_index(drop=True)
    X_trva_df = trval[feat_keep].copy()
    y_trva = trval["y_binary"].values.astype(int)

    test_df = df_model[test_mask]
    X_test_df = test_df[feat_keep].copy()
    y_test = test_df["y_binary"].values.astype(int)

    # 2) Identify numeric/categorical subset for this ablation (order preserved)
    num_keep = [c for c in feat_keep if (c in df_model.columns and pd.api.types.is_numeric_dtype(df_model[c]))]
    cat_keep = [c for c in feat_keep if c not in num_keep]

    # 3) Fit preprocessors on TRAIN ONLY (leakage-safe)
    X_train_fit = df_model[train_mask][feat_keep].copy()
    pre_tree = pre_tree_factory(num_keep, cat_keep)
    pre_scaled = pre_scaled_factory(num_keep, cat_keep)

    pre_tree = get_fitted_preprocessor(pre_tree, X_train_fit)
    pre_scaled = get_fitted_preprocessor(pre_scaled, X_train_fit)

    # 4) Transform tr+val and test
    X_trva_tree = pre_tree.transform(X_trva_df)
    X_trva_scaled = pre_scaled.transform(X_trva_df)
    Xte_tree = pre_tree.transform(X_test_df)
    Xte_scaled = pre_scaled.transform(X_test_df)

    # 5) Choose the right representation per model
    if model_name in ["LogReg", "MLP"]:
        X_trva = X_trva_scaled; Xte = Xte_scaled; pre_used = pre_scaled
    else:
        X_trva = X_trva_tree;   Xte = Xte_tree;   pre_used = pre_tree

    # 6) Fit with rolling time-series CV (train+val), refit on all tr+val
    cv_metrics, final_est = fit_with_cv(
        model_name=model_name, estimator=base_estimator,
        X=X_trva, y=y_trva, dates=dates_trva_full,
        task="binary", param_grid=None
    )

    # 7) Evaluate on test
    if hasattr(final_est, "predict_proba"):
        y_proba = final_est.predict_proba(Xte)[:, 1]
    elif hasattr(final_est, "decision_function"):
        scores = final_est.decision_function(Xte)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    else:
        y_proba = np.zeros_like(y_test, dtype=float)
    y_pred = final_est.predict(Xte)
    test_metrics = metric_summary_binary(y_test, y_pred, y_proba)

    # 8) Compute 95% CI from CV
    ci_row = {}
    for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        if m in cv_metrics:
            mean, (low, high) = ci_from_folds(cv_metrics[m])
            ci_row[f"cv_{m}_mean"] = mean
            ci_row[f"cv_{m}_ci_low"] = low
            ci_row[f"cv_{m}_ci_high"] = high

    # 9) Return a row
    out = {"model": model_name, "ablation": "ALL" if len(drops)==0 else f"DROP_{len(drops)}",
           "features_kept": len(feat_keep), "features_dropped": len(drops)}
    out.update({f"test_{k}": v for k, v in test_metrics.items()})
    out.update(ci_row)
    out["ablation_name"] = "ALL" if len(drops) == 0 else "NO_" + "_".join(sorted(set([blk for blk in []])))

    return out, final_est, pre_used


In [78]:
# ---------- 4) Top-level ablation runner ----------
def run_ablation_suite_binary(
    df, avg_col, pct_cols, num_cols, cat_cols,
    train_mask, val_mask, test_mask,
    dates_trva_full, cw_bin,
    models_to_run=("LogReg", "MLP", "RF", "STACK_BIN")
):
    """
    Produces:
     - artifacts/ablation_binary_results.csv
     - figs/ablation_* plots
    """
    # Full feature list
    features_all = num_cols + cat_cols
    df_model = df[features_all + ["y_binary", "_date"]].copy()

    # Define preprocessors factories
    def pre_tree_factory(num_keep, cat_keep):
        return make_preprocessor(num_cols=num_keep, cat_cols=cat_keep, scale_numeric=False)

    def pre_scaled_factory(num_keep, cat_keep):
        return make_preprocessor(num_cols=num_keep, cat_cols=cat_keep, scale_numeric=True)

    # Detect blocks and make scenarios
    blocks = detect_feature_blocks(df, avg_col, pct_cols, num_cols, cat_cols)
    scenarios = build_ablation_scenarios(blocks, features_all)

    # Build baseline models
    model_dict = {}
    # Binary baseline models (reuse your helper builder for consistency)
    baseline_bin = model_list_binary(class_weight_dict=cw_bin)

    # Keep only the requested models
    for k in ["LogReg", "MLP", "RF"]:
        if k in models_to_run and k in baseline_bin:
            model_dict[k] = baseline_bin[k]

    # Prepare storage
    rows = []
    fitted_store = {}  # (model, scenario_name) -> (estimator, preprocessor)

    for sc in scenarios:
        drop_cols = sc["drop"]
        sc_name = sc["name"]
        print(f"\n=== Scenario: {sc_name} ===")
        for model_name, est in model_dict.items():
            res = run_one_ablation_binary(
                model_name=model_name,
                base_estimator=est,
                features_all=features_all,
                drops=drop_cols,
                df_model=df_model,
                train_mask=train_mask, val_mask=val_mask, test_mask=test_mask,
                dates_trva_full=dates_trva_full,
                cw_bin=cw_bin,
                pre_tree_factory=pre_tree_factory,
                pre_scaled_factory=pre_scaled_factory
            )
            if res is None:
                print(f"[skip] {model_name} produced empty features after dropping {sc_name}")
                continue
            row, fitted_est, pre_used = res
            row["ablation_label"] = sc_name
            row["model"] = model_name
            rows.append(row)
            fitted_store[(model_name, sc_name)] = (fitted_est, pre_used)

        # STACK_BIN per scenario (if requested)
        if "STACK_BIN" in models_to_run:
            # Build base learners that are present
            base_learners = {}
            for base_name in ["RF", "LGBM", "XGB", "CAT"]:
                if base_name in baseline_bin:
                    base_learners[base_name] = baseline_bin[base_name]

            if base_learners:
                # Recreate features+preprocessors (same as run_one_ablation_binary, short form)
                feat_keep = [c for c in features_all if c not in drop_cols]
                if len(feat_keep) == 0:
                    print("[skip] STACK_BIN empty feat set")
                else:
                    X_train_fit = df_model[train_mask][feat_keep].copy()
                    pre_tree = pre_tree_factory(
                        [c for c in feat_keep if pd.api.types.is_numeric_dtype(df_model[c])],
                        [c for c in feat_keep if not pd.api.types.is_numeric_dtype(df_model[c])]
                    )
                    pre_tree = get_fitted_preprocessor(pre_tree, X_train_fit)

                    trval = df_model[train_mask | val_mask].sort_values("_date").reset_index(drop=True)
                    X_trva_tree = pre_tree.transform(trval[feat_keep])
                    y_trva = trval["y_binary"].values.astype(int)
                    Xte_tree = pre_tree.transform(df_model[test_mask][feat_keep])
                    y_test = df_model[test_mask]["y_binary"].values.astype(int)

                    meta_lr = LogisticRegression(max_iter=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
                    stack = stacked_ensemble(base_learners, meta_lr)

                    # CV (tree rep)
                    cv_metrics, stack_fitted = fit_with_cv(
                        model_name="STACK_BIN", estimator=stack,
                        X=X_trva_tree, y=y_trva, dates=dates_trva_full,
                        task="binary", param_grid=None
                    )

                    # Test eval
                    if hasattr(stack_fitted, "predict_proba"):
                        y_proba = stack_fitted.predict_proba(Xte_tree)[:, 1]
                    else:
                        scores = stack_fitted.decision_function(Xte_tree)
                        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
                    y_pred = stack_fitted.predict(Xte_tree)
                    tm = metric_summary_binary(y_test, y_pred, y_proba)
                    ci_row = {}
                    for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
                        if m in cv_metrics:
                            mean, (low, high) = ci_from_folds(cv_metrics[m])
                            ci_row[f"cv_{m}_mean"] = mean
                            ci_row[f"cv_{m}_ci_low"] = low
                            ci_row[f"cv_{m}_ci_high"] = high

                    row = {"model": "STACK_BIN", "ablation_label": sc_name,
                           "features_kept": len(feat_keep), "features_dropped": len(drop_cols)}
                    row.update({f"test_{k}": v for k, v in tm.items()})
                    row.update(ci_row)
                    rows.append(row)
                    fitted_store[("STACK_BIN", sc_name)] = (stack_fitted, pre_tree)

    # Aggregate results
    res_df = pd.DataFrame(rows)
    out_csv = ARTIFACT_DIR / "ablation_binary_results.csv"
    res_df.to_csv(out_csv, index=False)
    print("Ablation results saved →", out_csv)

    # ------- Plots: F1 vs scenario by model -------
    if not res_df.empty:
        # Ensure consistent ordering (ALL first)
        res_df["scenario_order"] = res_df["ablation_label"].apply(lambda s: 0 if s=="ALL" else 1)
        res_df = res_df.sort_values(["scenario_order", "ablation_label", "model"])

        # F1 bar per model
        for mdl in res_df["model"].unique():
            sub = res_df[res_df["model"] == mdl].copy()
            plt.figure(figsize=(10, 4))
            sns.barplot(x="ablation_label", y="test_f1", data=sub, color="steelblue")
            plt.title(f"Ablation — Binary F1 (Test) — {mdl}")
            plt.xlabel("Scenario")
            plt.ylabel("F1")
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"ablation_binary_f1_{mdl}.png", dpi=300)
            plt.close()

        # Delta vs ALL baseline
        deltas = []
        for mdl in res_df["model"].unique():
            base_f1 = res_df[(res_df["model"] == mdl) & (res_df["ablation_label"] == "ALL")]["test_f1"].mean()
            tmp = res_df[res_df["model"] == mdl].copy()
            tmp["delta_f1_vs_all"] = tmp["test_f1"] - base_f1
            deltas.append(tmp)
        deltas = pd.concat(deltas, axis=0)

        plt.figure(figsize=(12, 5))
        sns.barplot(x="ablation_label", y="delta_f1_vs_all", hue="model", data=deltas)
        plt.axhline(0, color="k", linewidth=1)
        plt.title("Ablation — ΔF1 vs ALL baseline (Test)")
        plt.xlabel("Scenario")
        plt.ylabel("ΔF1")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "ablation_binary_delta_f1.png", dpi=300)
        plt.close()

    return res_df


In [79]:
# Reuse your earlier computed sets
features_all = []  # will be computed inside the function
# num_cols & cat_cols come from your earlier 'get_feature_groups'
# df must already include engineered features & 'y_binary'


In [80]:
# Build combined Train+Val time index
df_model_for_dates = df[num_cols + cat_cols + ["y_binary", "_date"]]
df_train_val = df_model_for_dates[train_mask | val_mask].sort_values("_date").reset_index(drop=True)
dates_trva_full = df_train_val["_date"]

ablation_results_df = run_ablation_suite_binary(
    df=df, avg_col=avg_col, pct_cols=pct_cols, num_cols=num_cols, cat_cols=cat_cols,
    train_mask=train_mask, val_mask=val_mask, test_mask=test_mask,
    dates_trva_full=dates_trva_full, cw_bin=cw_bin,
    models_to_run=("LogReg", "MLP", "RF", "STACK_BIN")
)

display(ablation_results_df.head())



=== Scenario: ALL ===

=== Scenario: NO_TEMPORAL ===

=== Scenario: NO_PERCENT_BINS ===

=== Scenario: NO_LAGS ===

=== Scenario: NO_ROLLING ===

=== Scenario: NO_DERIVED ===

=== Scenario: NO_CATEGORICAL ===
Ablation results saved → artifacts/ablation_binary_results.csv


,model,ablation,features_kept,features_dropped,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,cv_accuracy_mean,...,cv_recall_ci_high,cv_f1_mean,cv_f1_ci_low,cv_f1_ci_high,cv_roc_auc_mean,cv_roc_auc_ci_low,cv_roc_auc_ci_high,ablation_name,ablation_label,scenario_order
0,LogReg,ALL,30,0,0.960875,0.944984,0.976269,0.960371,0.995395,0.971659,...,0.975516,0.951552,0.936884,0.966221,0.996609,0.995725,0.997492,ALL,ALL,0
1,MLP,ALL,30,0,0.953335,0.952925,0.950877,0.951900,0.992620,0.962611,...,0.959922,0.934992,0.914879,0.955105,0.993760,0.992470,0.995050,ALL,ALL,0
2,RF,ALL,30,0,0.944308,0.941379,0.944106,0.942741,0.989755,0.951969,...,0.936659,0.907028,0.869858,0.944198,0.991603,0.989569,0.993637,ALL,ALL,0
3,STACK_BIN,NaN,30,0,0.938064,0.958717,0.911716,0.934626,0.989755,0.944881,...,0.903552,0.900782,0.870267,0.931298,0.991603,0.989569,0.993637,NaN,ALL,0
24,LogReg,DROP_6,24,6,0.960875,0.944843,0.976432,0.960378,0.995478,0.971993,...,0.976906,0.952576,0.938833,0.966319,0.996705,0.995833,0.997577,NO_,NO_CATEGORICAL,1


In [81]:
# Example scenarios (align with your ablation setup)
# Be sure these 'drop' lists match your earlier block detection.
scenarios = [
    {"name": "ALL", "drop": []},
    {"name": "NO_TEMPORAL", "drop": [c for c in ["year","month","season","time_index"] if c in df.columns]},
    {"name": "NO_PERCENT_BINS", "drop": [c for c in pct_cols if c in df.columns]},
    {"name": "NO_LAGS", "drop": [c for c in df.columns if "_lag" in c]},
    {"name": "NO_ROLLING", "drop": [c for c in df.columns if "_roll" in c]},
    {"name": "NO_CATEGORICAL", "drop": [c for c in cat_cols if c in df.columns]},
    {"name": "NO_DERIVED", "drop": [c for c in ["pct_over_15", "pct_on_time", "traffic_intensity_proxy"] if c in df.columns]},
]

In [89]:
pip install shap

In [90]:
run_shap_explainability_for_ablation(
    df=df,
    avg_col=avg_col, pct_cols=pct_cols,
    num_cols=num_cols, cat_cols=cat_cols,
    train_mask=train_mask, val_mask=val_mask, test_mask=test_mask,
    dates_trva_full=dates_trva_full,
    cw_bin=cw_bin,
    scenarios=scenarios,
    models=("LogReg", "MLP", "RF", "STACK_BIN"),  # choose subset if needed
    max_test_samples=2000
)

SHAP not available. Please `pip install shap` and rerun.


In [93]:
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER
from pathlib import Path
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.4 MB/s eta 0:00:00


In [100]:
def generate_final_pdf_report(
        pdf_path,
        artifact_dir,
        figure_dir,
        title="Flight Delay Prediction Analysis Report",
        author="John Ogunlola"
    ):
    """
    Generate a complete PDF containing:
      - Title Page
      - Summary Tables (binary, multiclass)
      - All figures inside figure_dir
      - Any text summaries inside artifact_dir
    """

    artifact_dir = Path(artifact_dir)
    figure_dir = Path(figure_dir)
    pdf_path = Path(pdf_path)

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        name="TitleStyle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontSize=24,
        spaceAfter=20
    )
    heading_style = styles["Heading2"]
    body_style = styles["BodyText"]

    # Build Document
    doc = SimpleDocTemplate(
        str(pdf_path), # Convert Path object to string here
        pagesize=A4,
        rightMargin=40, leftMargin=40,
        topMargin=40, bottomMargin=40
    )
    story = []

    # ------------------------------------------
    #   TITLE PAGE
    # ------------------------------------------
    story.append(Paragraph(title, title_style))
    story.append(Paragraph(f"Author: {author}", body_style))
    story.append(Paragraph("Generated Automatically", body_style))
    story.append(Spacer(1, 0.5 * inch))
    story.append(PageBreak())

    # ------------------------------------------
    #   TEXT SUMMARIES (CSV → Text)
    # ------------------------------------------
    summary_files = [
        "binary_test_scores.csv",
        "multiclass_test_scores.csv",
        "binary_cv_confidence_intervals.csv",
        "multiclass_cv_confidence_intervals.csv",
        "ablation_binary_results.csv",  # if exists
        "summary.txt"
    ]

    for fname in summary_files:
        fpath = artifact_dir / fname
        if fpath.exists():
            story.append(Paragraph(f"<b>{fname}</b>", heading_style))
            try:
                txt = fpath.read_text()
                # Escape < >
                txt = txt.replace("<", "&lt;").replace(">", "&gt;")
                lines = txt.split("\n")
                for ln in lines:
                    story.append(Paragraph(ln, body_style))
                story.append(Spacer(1, 0.3 * inch))
                story.append(PageBreak())
            except Exception as e:
                story.append(Paragraph(f"Could not load {fname}: {e}", body_style))

    # ------------------------------------------
    #   FIGURES SECTION
    # ------------------------------------------
    story.append(Paragraph("<b>Figures</b>", title_style))
    story.append(Spacer(1, 0.2 * inch))

    all_pngs = sorted(list(figure_dir.glob("*.png")))

    # If there are subfolders (e.g., ablation_shap/)
    for sub in figure_dir.iterdir():
        if sub.is_dir():
            all_pngs += sorted(list(sub.glob("*.png")))

    if not all_pngs:
        story.append(Paragraph("No PNG figures found.", body_style))

    for img_path in all_pngs:
        story.append(Paragraph(f"<b>{img_path.name}</b>", heading_style))
        try:
            im = Image(str(img_path), width=5.5*inch, height=4.0*inch)
            story.append(im)
            story.append(Spacer(1, 0.3 * inch))
        except Exception as e:
            story.append(Paragraph(f"Could not embed image {img_path}: {e}", body_style))

        story.append(PageBreak())

    # ------------------------------------------
    #   BUILD PDF
    # ------------------------------------------
    doc.build(story)
    print(f"PDF report saved to: {pdf_path.resolve()}")

In [101]:
generate_final_pdf_report(
    pdf_path="final_analysis_report.pdf",
    artifact_dir=ARTIFACT_DIR,
    figure_dir=FIG_DIR
)

PDF report saved to: /content/final_analysis_report.pdf
